In [2]:
import json
import urllib.request
from datetime import date, timedelta

# Endpoint público do OpenRouter (não consome cota nem precisa de API Key)
URL_CATALOGO = "https://openrouter.ai/api/v1/models"

print("Consultando catálogo oficial do OpenRouter...")
with urllib.request.urlopen(URL_CATALOGO, timeout=30) as resposta:
    dados = json.load(resposta)
    catalogo = dados.get("data", [])

# Filtro de expiração: ignora modelos de teste que saem do ar nos próximos 30 dias
data_corte = (date.today() + timedelta(days=30)).isoformat()

# Filtra apenas modelos viáveis para o nosso Text-to-SQL
modelos_aptos = []
for m in catalogo:
    modelo_id = m.get("id", "")
    parametros_suportados = m.get("supported_parameters") or []
    data_expiracao = m.get("expiration_date")

    # 1. Deve ser gratuito (:free)
    eh_gratuito = modelo_id.endswith(":free")
    
    # 2. Deve suportar Tool Calling (obrigatório para @agent.tool do PydanticAI)
    suporta_tools = "tools" in parametros_suportados
    
    # 3. Não pode expirar no curto prazo
    nao_expira_cedo = not (data_expiracao and data_expiracao <= data_corte)

    if eh_gratuito and suporta_tools and nao_expira_cedo:
        modelos_aptos.append(m)

# Exibe o resultado formatado para seleção
print(f"\nEncontrados {len(modelos_aptos)} modelos gratuitos com suporte a ferramentas:\n")
print(f"{'#':<3} | {'ID DO MODELO (Copie este valor)':<50} | {'JANELA DE CONTEXTO'}")
print("-" * 80)

for idx, m in enumerate(modelos_aptos, start=1):
    contexto = f"{m.get('context_length', 0):,} tokens"
    # O 'm["id"]' é o texto exato que vai no .env e no config.py
    print(f"{idx:<3} | {m['id']:<50} | {contexto}")

Consultando catálogo oficial do OpenRouter...

Encontrados 14 modelos gratuitos com suporte a ferramentas:

#   | ID DO MODELO (Copie este valor)                    | JANELA DE CONTEXTO
--------------------------------------------------------------------------------
1   | apodex/apodex-1.1-mini:free                        | 262,144 tokens
2   | inclusionai/ling-3.0-flash-sante:free              | 262,144 tokens
3   | qwen/qwen3.8-27b:free                              | 262,144 tokens
4   | dots-studio/dots-3-note-preview:free               | 512,000 tokens
5   | liquid/lfm-2.5-2.6b:free                           | 65,536 tokens
6   | nvidia/nemotron-3.5-lightning:free                 | 1,000,000 tokens
7   | thinkingmachines/inkling-small:free                | 1,048,576 tokens
8   | thinkingmachines/inkling:free                      | 1,048,576 tokens
9   | cohere/north-mini-code:free                        | 256,000 tokens
10  | nvidia/nemotron-3-ultra-550b-a55b:free             | 1,0

In [ ]:
import json
import os
import time
import urllib.request

from dotenv import load_dotenv

# Carrega a OPENROUTER_API_KEY do seu arquivo .env
load_dotenv()
CANDIDATOS = [
    #"qwen/qwen3.8-27b:free",
    #"cohere/north-mini-code:free",
    #"google/gemma-4-31b-it:free",
    #"nvidia/nemotron-3-super-120b-a12b:free",
    #"nvidia/nemotron-3.5-lightning:free",
]

# Definição técnica da ferramenta no formato padrão do OpenAI / OpenRouter
ESQUEMA_TOOL = {
    "type": "function",
    "function": {
        "name": "executar_sql",
        "description": "Executa uma consulta SQL SELECT no banco CineData.",
        "parameters": {
            "type": "object",
            "properties": {
                "sql": {
                    "type": "string",
                    "description": "A consulta SQL a ser executada."
                }
            },
            "required": ["sql"]
        }
    }
}

def testar_suporte_tool_calling(modelo_id: str) -> str:
    """Envia uma requisição de teste para verificar se o modelo gera 'tool_calls'."""
    corpo_requisicao = {
        "model": modelo_id,
        "max_tokens": 300,
        "tools": [ESQUEMA_TOOL],
        "tool_choice": "auto",
        "messages": [
            {
                "role": "user",
                "content": "Use a ferramenta executar_sql para contar as linhas da tabela dim_movies."
            }
        ]
    }
    
    chave = os.getenv("OPENROUTER_API_KEY", "")
    if not chave:
        return "ERRO: OPENROUTER_API_KEY não configurada no .env"
    
    headers = {
        "Authorization": f"Bearer {chave}",
        "Content-Type": "application/json",
        "HTTP-Referer": "http://localhost:8501",  # Identificador para o OpenRouter
        "X-Title": "CineData Agent Test"
    }
    
    req = urllib.request.Request(
        "https://openrouter.ai/api/v1/chat/completions",
        data=json.dumps(corpo_requisicao).encode("utf-8"),
        headers=headers
    )
    
    try:
        with urllib.request.urlopen(req, timeout=45) as resposta:
            dados = json.load(resposta)
            escolha = dados["choices"][0]["message"]
            
            # Validação: o modelo realmente acionou a ferramenta?
            if escolha.get("tool_calls"):
                return "OK (Tool call acionada com sucesso)"
            else:
                return "FALHA (Respondeu texto puro, ignorou a tool)"
                
    except urllib.error.HTTPError as e:
        return f"ERRO HTTP {e.code}: {e.reason}"
    except Exception as e:
        return f"ERRO: {e!s}"

# Execução do teste em lote
aprovados = []

print("Iniciando testes de compatibilidade com Tool Calling...\n")
for modelo in CANDIDATOS:
    print(f"Testando: {modelo} ...", end=" ", flush=True)
    status = testar_suporte_tool_calling(modelo)
    print(status)
    
    if status.startswith("OK"):
        aprovados.append(modelo)
        
    # Intervalo de 3 segundos para evitar bloqueios de rate limit (429) no OpenRouter
    time.sleep(3)

print("\n" + "=" * 80)
print(f"Resultado: {len(aprovados)} de {len(CANDIDATOS)} modelos foram aprovados.")
print("=" * 80)

if aprovados:
    print("\nCopie a linha abaixo e cole no seu arquivo .env:")
    print(f"MODELOS_OPENROUTER={','.join(aprovados)}")
else:
    print("\nNenhum modelo foi aprovado. Verifique sua chave ou escolha outros candidatos.")

Iniciando testes de compatibilidade com Tool Calling...

Testando: qwen/qwen3.8-27b:free ... OK (Tool call acionada com sucesso)
Testando: cohere/north-mini-code:free ... OK (Tool call acionada com sucesso)
Testando: google/gemma-4-31b-it:free ... ERRO HTTP 429: Too Many Requests
Testando: nvidia/nemotron-3-super-120b-a12b:free ... OK (Tool call acionada com sucesso)
Testando: nvidia/nemotron-3.5-lightning:free ... OK (Tool call acionada com sucesso)

Resultado: 4 de 5 modelos foram aprovados.

Copie a linha abaixo e cole no seu arquivo .env:
MODELOS_OPENROUTER=qwen/qwen3.8-27b:free,cohere/north-mini-code:free,nvidia/nemotron-3-super-120b-a12b:free,nvidia/nemotron-3.5-lightning:free
